# Logarithmic Transformations & Quadratic Terms. Lecture Notebook
### Applied Statistical Data Analysis. Prof. Dr. Kristyna Ters | MSc Finance | FHNW
**Based on:** Brooks, C., *Introductory Econometrics for Finance*, Cambridge University Press, Ch. 4-5

---
**Learning Objectives:**
- Interpret coefficients in all **four log forms** (elasticities, semi-elasticities)
- Read **dummy coefficients in log models** correctly: exact effect $100(e^{\delta}-1)\%$
- Model nonlinear effects with **quadratic terms**: marginal effects and the turning point
- Choose between functional forms honestly (theory first, RESET as referee)

> Run each cell with **Shift+Enter**. This notebook accompanies the V10 lecture slides.
> The slides use illustrative values; the code computes live (or seeded) numbers. They will be close, but not identical.

## Step 0: Install & Import Libraries

In [ ]:
!pip install yfinance statsmodels --quiet

import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.diagnostic import linear_reset, het_breuschpagan
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'white', 'axes.facecolor':'white',
    'axes.spines.top':False, 'axes.spines.right':False,
    'axes.grid':True, 'grid.alpha':0.3, 'font.size':11
})
YELLOW = '#FDE70E'; ORANGE = '#FCB310'; RED = '#C70101'
GREY   = '#4B4B4B'; BLUE = '#0E75FE'; GREEN = '#0B7A3C'
print('✓ Libraries loaded.')

---
# Part 1: The Swiss Cross-Section (Elasticity + SMI Dummy)

We use a cross-section of 50 Swiss listed companies: average daily trading volume in CHF and market
capitalisation, plus a dummy for SMI membership.

The data are a **frozen snapshot** that lives in the course repository, so every run of this notebook
reproduces the figures on the slides exactly. That matters here: market capitalisation is a
point-in-time quantity and a rolling volume window moves every day, so a live download would give
you different coefficients from the ones in the video. The loader falls back to a live download if
the file cannot be reached, and tells you which path it took.

### 1.1 Load the cross-section

In [ ]:
CSV = ('https://raw.githubusercontent.com/KristynaTers/ASDA/main/data/'
       'ASDA_SwissVolumeSize_2026.csv')

# market capitalisation as at 17.09.2026; average daily turnover (close x volume)
# over the 126 trading days from 02.03.2026 to 31.08.2026
try:
    raw = pd.read_csv(CSV, comment='#')
    df = (raw.rename(columns={'mcap_chf': 'mcap', 'turnover_chf': 'volume'})
             .set_index('ticker')[['mcap', 'volume', 'D_SMI']].astype(float))
    source = 'FROZEN course dataset (reproduces the slides)'
except Exception as e:
    print(f'could not read the course dataset ({type(e).__name__}), falling back to Yahoo')
    SMI = ['NESN.SW', 'NOVN.SW', 'UBSG.SW', 'ZURN.SW', 'CFR.SW', 'ABBN.SW', 'SIKA.SW',
           'LONN.SW', 'ALC.SW', 'GIVN.SW', 'HOLN.SW', 'SLHN.SW', 'PGHN.SW', 'SCMN.SW',
           'SREN.SW', 'GEBN.SW', 'SOON.SW', 'LOGN.SW', 'KNIN.SW']
    MID = ['SCHP.SW', 'LISP.SW', 'EMSN.SW', 'SGSN.SW', 'VACN.SW', 'STMN.SW', 'BAER.SW',
           'BCVN.SW', 'PSPN.SW', 'BKW.SW', 'FHZN.SW', 'BARN.SW', 'AVOL.SW', 'BANB.SW',
           'SFSN.SW', 'SIGN.SW', 'SUN.SW', 'TEMN.SW', 'GF.SW', 'DKSH.SW', 'ADEN.SW',
           'GALE.SW', 'CLN.SW', 'ALLN.SW', 'BUCN.SW', 'CMBN.SW', 'MOBN.SW', 'KARN.SW',
           'OERL.SW', 'BELL.SW', 'IMPN.SW']
    rows, dropped = [], []
    for tick in SMI + MID:
        try:
            t = yf.Ticker(tick)
            px = t.history(period='6mo')
            mcap = t.fast_info['marketCap']
            if len(px) < 60 or not mcap or mcap <= 0:
                dropped.append(tick); continue
            rows.append({'ticker': tick,
                         'mcap': float(mcap),
                         'volume': float((px['Close'] * px['Volume']).mean()),
                         'D_SMI': 1.0 if tick in SMI else 0.0})
        except Exception:
            dropped.append(tick)
    if not rows:
        raise RuntimeError('neither the course dataset nor Yahoo is reachable')
    if dropped:
        print(f'{len(dropped)} tickers dropped: {dropped}')
    df = pd.DataFrame(rows).set_index('ticker')
    source = 'LIVE Yahoo download (your numbers will differ from the slides)'

print(f'{source}')
print(f'n = {len(df)} stocks ({int(df.D_SMI.sum())} SMI, {int((1 - df.D_SMI).sum())} SMI MID)')
print(f'market cap  : {df["mcap"].min()/1e9:6.1f} to {df["mcap"].max()/1e9:6.1f} bn CHF'
      f'   (factor {df["mcap"].max()/df["mcap"].min():.0f})')
print(f'turnover    : {df["volume"].min()/1e6:6.1f} to {df["volume"].max()/1e6:6.1f} m CHF per day')
df.head(3)

### 1.2 Why logs: the skewness of levels

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.3))
axes[0].hist(df['mcap']/1e9, bins=20, color=YELLOW, edgecolor=GREY, lw=0.4)
axes[0].set_title('Market caps in levels: right-skewed', fontweight='bold', loc='left')
axes[0].set_xlabel('market cap (bn)')
axes[1].hist(np.log(df['mcap']), bins=14, color=YELLOW, edgecolor=GREY, lw=0.4)
axes[1].set_title('In logs: roughly symmetric', fontweight='bold', loc='left')
axes[1].set_xlabel('ln(market cap)')
plt.tight_layout(); plt.show()

---
# Part 2: The Elasticity (log-log)

$$\ln V_i = \beta_0 + \beta_1 \ln M_i + \delta D_i^{SMI} + u_i$$

In the log-log form, $\beta_1$ is the **elasticity**: a 1% larger market cap goes with $\beta_1$% larger volume. Scale-free: no currencies, no billions.

In [ ]:
df['ln_vol']  = np.log(df['volume'])
df['ln_mcap'] = np.log(df['mcap'])

X = sm.add_constant(df[['ln_mcap', 'D_SMI']])
m = sm.OLS(df['ln_vol'], X).fit(cov_type='HC1')

print(f'elasticity (ln_mcap): {m.params["ln_mcap"]:.3f}  (SE {m.bse["ln_mcap"]:.3f}, '
      f't = {m.tvalues["ln_mcap"]:.1f})')
print(f'D_SMI:                {m.params["D_SMI"]:.3f}  (SE {m.bse["D_SMI"]:.3f}, '
      f't = {m.tvalues["D_SMI"]:.2f}, p = {m.pvalues["D_SMI"]:.3f})')
print(f'R² = {m.rsquared:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.8))
axes[0].scatter(df['mcap']/1e9, df['volume']/1e6, s=18, color=GREY, alpha=0.6)
axes[0].set_xlabel('market cap (bn)'); axes[0].set_ylabel('daily volume (m)')
axes[0].set_title('Levels: curved, dominated by the giants', fontweight='bold', loc='left')

cols = np.where(df['D_SMI'] > 0, RED, BLUE)
axes[1].scatter(df['ln_mcap'], df['ln_vol'], s=18, c=cols, alpha=0.7)
b = np.polyfit(df['ln_mcap'], df['ln_vol'], 1)
xx = np.linspace(df['ln_mcap'].min(), df['ln_mcap'].max(), 40)
axes[1].plot(xx, np.polyval(b, xx), color='black', lw=1.8)
axes[1].set_xlabel('ln(market cap)'); axes[1].set_ylabel('ln(volume)')
axes[1].set_title(f'Log-log: a straight line, unconditional slope = {b[0]:.2f}',
                  fontweight='bold', loc='left')
# NOTE: this is the UNCONDITIONAL fit. The elasticity reported above comes from the
# model that also holds D_SMI fixed, so the two slopes need not be the same number.
plt.tight_layout(); plt.show()

---
# Part 3: The Dummy in a Log Model (the Exact Effect)

The naive reading "$\delta = 0.62$ means +62%" is wrong: a dummy is a discrete jump, not a small change. The exact percentage effect is

$$100 \cdot (e^{\hat{\delta}} - 1)\%.$$

In [ ]:
delta = m.params['D_SMI']
naive = 100 * delta
exact = 100 * (np.exp(delta) - 1)
rev   = 100 * (np.exp(-delta) - 1)

print(f'delta_hat            = {delta:.3f}')
print(f'naive reading        = {naive:+.1f}%   (wrong for large |delta|)')
print(f'exact effect         = {exact:+.1f}%   (SMI members vs comparable mid caps)')
print(f'reversed dummy       = {rev:+.1f}%   (mid caps vs comparable SMI members)')
print('\nNote the asymmetry: the up and down percentage effects are not mirror images,')
print('exactly like returns: +50% followed by -50% does not bring you back.')

**Rule of thumb:** for $|\delta| < 0.10$ the naive reading is fine ($e^{0.10}-1 = 10.5\%$). Beyond that, always report the exact effect. Everything else about dummies (m − 1 rule, reference category, t-test) is unchanged from the dummy-variables chapter.

---
# Part 4: Choosing the Form: What RESET Can and Cannot Settle

Within the same dependent variable, RESET flags the form that misses **curvature**. We run it on the
levels and on the log-log specification, and then, because RESET is not the only diagnostic that bears
on the choice, we look at the error variance as well. (Reminder: never compare R² across different y.)

In [ ]:
# The levels regression must be run in SCALED units. In raw CHF the fitted values
# are of order 1e8, so RESET's auxiliary regressors y_hat^2 and y_hat^3 reach 1e16
# and 1e24: the auxiliary design becomes numerically rank-deficient, statsmodels
# drops one restriction and reports F with 1 numerator degree of freedom instead of
# the q = 2 this chapter defines. Always check df_num.
lvl = pd.DataFrame({'volume_m': df['volume'] / 1e6,     # millions of CHF
                    'mcap_bn':  df['mcap'] / 1e9,       # billions of CHF
                    'D_SMI':    df['D_SMI']})
X_lvl = sm.add_constant(lvl[['mcap_bn', 'D_SMI']])
m_lvl = sm.OLS(lvl['volume_m'], X_lvl).fit()

r_lvl = linear_reset(m_lvl, power=3, use_f=True)
r_log = linear_reset(m, power=3, use_f=True)
F_crit = stats.f.ppf(0.95, 2, int(r_lvl.df_denom))

print('RESET, H0: no omitted curvature')
for name, r in (('levels ', r_lvl), ('log-log', r_log)):
    v = float(np.squeeze(r.fvalue))
    print(f'  {name}: F = {v:6.2f}, df = ({int(r.df_num)}, {int(r.df_denom)}), '
          f'p = {r.pvalue:.3f}  ->  {"REJECT" if v > F_crit else "do not reject"}')
print(f'  5% critical value F(2, {int(r_lvl.df_denom)}) = {F_crit:.2f}')

# RESET tests the MEAN function. The variance is a separate question, and on this
# cross-section it is the one that decides.
bp_lvl = het_breuschpagan(m_lvl.resid, X_lvl)
bp_log = het_breuschpagan(m.resid,     m.model.exog)
chi2c  = stats.chi2.ppf(0.95, 2)
print('\nBreusch-Pagan, H0: homoskedasticity')
print(f'  levels : LM = {bp_lvl[0]:6.2f}, p = {bp_lvl[1]:.4f}  ->  '
      f'{"REJECT" if bp_lvl[0] > chi2c else "do not reject"}')
print(f'  log-log: LM = {bp_log[0]:6.2f}, p = {bp_log[1]:.4f}  ->  '
      f'{"REJECT" if bp_log[0] > chi2c else "do not reject"}')
print(f'  5% critical value chi2(2) = {chi2c:.2f}')

# how the levels residuals fan out with size
lvl = lvl.assign(res=m_lvl.resid)
ter = lvl.assign(g=pd.qcut(lvl['mcap_bn'], 3, labels=['small', 'middle', 'large']))
print('\nStandard deviation of the LEVELS residuals by size tercile (m CHF per day):')
for g, sub in ter.groupby('g', observed=True):
    print(f'  {g:<7} mcap {sub["mcap_bn"].min():6.1f} to {sub["mcap_bn"].max():6.1f} bn'
          f'   sd(residual) = {sub["res"].std():6.2f}')

# Let the tests speak: never print a verdict the output above can contradict.
reset_picks = (float(np.squeeze(r_lvl.fvalue)) > F_crit) != (float(np.squeeze(r_log.fvalue)) > F_crit)
bp_picks    = (bp_lvl[0] > chi2c) != (bp_log[0] > chi2c)
print('\nReading:')
print(f'  RESET {"separates the two forms" if reset_picks else "rejects neither form, so it does not pick the winner"}.')
print(f'  Breusch-Pagan {"separates them: the levels errors fan out with firm size, in logs that is gone" if bp_picks else "does not separate them either"}.')
print('  Whatever the tests say, the case for logs also rests on the economics')
print('  (volume scales multiplicatively with size) and on interpretability:')
print('  the log-log slope is an elasticity and is free of units.')

---
# Part 5: Quadratic Terms (Fund Size and Performance)

Fund-level alpha data are proprietary, so we simulate a realistic sample of 120 equity funds, calibrated to magnitudes from the fund-performance literature, with a fixed seed so everyone gets identical numbers. The data-generating process has an inverted-U shape by construction; the exercise is to recover and interpret it.

$$alpha_i = \beta_0 + \beta_1\, size_i + \beta_2\, size_i^2 + u_i$$

In [ ]:
rng = np.random.default_rng(42)
funds = pd.DataFrame({'size': rng.uniform(0.05, 2.2, 120)})            # size in bn CHF
funds['alpha'] = -0.2 + 2.4*funds['size'] - 1.5*funds['size']**2 \
                 + rng.normal(0, 0.55, 120)                             # alpha in % p.a.

funds['size2'] = funds['size']**2
Xq = sm.add_constant(funds[['size', 'size2']])
mq = sm.OLS(funds['alpha'], Xq).fit(cov_type='HC1')

b1, b2 = mq.params['size'], mq.params['size2']
print(f'size:   {b1:+.3f}  (t = {mq.tvalues["size"]:.1f})')
print(f'size²:  {b2:+.3f}  (t = {mq.tvalues["size2"]:.1f})')
print('→ positive first, negative second: an inverted U')

### 5.1 Turning point and marginal effects

In [ ]:
xstar = -b1 / (2*b2)
print(f'turning point: x* = -({b1:.3f}) / (2 · ({b2:.3f})) = {xstar:.2f} bn CHF')
print(f'inside the data range [{funds["size"].min():.2f}, {funds["size"].max():.2f}]? '
      f'{"yes → evidence" if funds["size"].min() < xstar < funds["size"].max() else "NO → extrapolation!"}')

for x0 in (0.2, xstar, 1.5):
    meff = b1 + 2*b2*x0
    print(f'marginal effect at size = {x0:.2f} bn: {meff:+.2f} alpha points per additional bn')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(funds['size'], funds['alpha'], s=14, color=GREY, alpha=0.55, label='funds')
xx = np.linspace(0.03, 2.25, 100)
yy = mq.params['const'] + b1*xx + b2*xx**2
ax.plot(xx, yy, color=RED, lw=2.2, label='fitted quadratic')
ax.axvline(xstar, color=BLUE, ls=':', lw=1.5)
ax.set_xlabel('fund size (bn CHF)'); ax.set_ylabel('alpha (% p.a.)')
ax.set_title('Rising, peaking, declining: the inverted U', fontweight='bold', loc='left')
ax.legend(frameon=False)
plt.tight_layout(); plt.show()
print('Reporting rule: never report THE effect of size. Report marginal effects at')
print('several relevant levels plus the turning point. That is the honest summary of a curve.')

---
## Summary Table

| Form | Model | Reading of β₁ |
|------|-------|----------------|
| lin-lin | y on x | β₁ units of y per unit of x |
| log-lin | ln(y) on x | about 100·β₁ % of y per unit of x (semi-elasticity) |
| lin-log | y on ln(x) | β₁/100 units of y per 1% of x |
| log-log | ln(y) on ln(x) | β₁ % of y per 1% of x (**elasticity**) |

| Situation | Rule |
|-----------|------|
| Dummy in a log model | exact effect $100(e^{\delta}-1)\%$; naive 100·δ only for \|δ\| < 0.10 |
| Quadratic | marginal effect $\beta_1 + 2\beta_2 x$; turning point $x^* = -\beta_1/(2\beta_2)$, only inside the data range |
| Model comparison | theory first; never compare R² across different y; RESET as referee |

---
*Applied Statistical Data Analysis | Prof. Dr. Kristyna Ters | FHNW School of Business | HS 2026*

*Next: Binary Dependent Variables (LPM, Logit & Probit, Maximum Likelihood).*